<a href="https://colab.research.google.com/github/CharlieLaughton/crossflow/blob/master/Crossflow_101.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tool calling from Python with Crossflow

This workbook illustrates how the `crossflow` Python package can be used to interface between a Python script and some external tool that has a command line interface.

Reference: [Cox et al., "Crossflow: A Python Library for Computational Chemistry Workflows"](https://openresearchsoftware.metajnl.com/articles/10.5334/jors.539)


In essence, Crossflow allows tools run from a command line:
```bash
% mytool -i inputfile -o outputfile
```
to become callable functions in Python ("tasks"):
```python
outputfile = mytool_task(inputfile)
```
----------------
## Set up
Begin by importing crossflow:


In [ ]:
!pip install crossflow

In [ ]:
import crossflow

We will use the unix utility `sort` as a dummy tool that we want to interface with. `Sort` sorts the lines in an input file and writes them to an output file:
```
% sort -i inputfile -o sortedfile
```
First let's create a file of text to work with:

In [ ]:
!STR=$'Hello\nWorld\nHow are\nyou today?'; echo "$STR" > inputfile.txt
!cat inputfile.txt

Now let's see how it works from the command line:

In [ ]:
!sort -i inputfile.txt -o sortedfile.txt
!cat sortedfile.txt

As expected, `sortedfile.txt` contains the lines in `inputfile.txt`, but in sorted order.


## First steps with crossflow

Now let's see how to create a `crossflow` task that allows you to do the same thing inside Python.

Here's the code, then after the explanation:

In [ ]:
from crossflow.tasks import SubprocessTask
sort_task = SubprocessTask('sort -i input -o output')
sort_task.set_inputs(['input'])
sort_task.set_outputs(['output'])

1. In the first line, we import the `SubprocessTask` class.
2. In the second line we create an instance of a task, using a *template string*. The *template string* needs to be a valid example of how your tool runs, but the actual names of inputs and outputs are just placeholders - they don't have to be the same as any real files you have - so `sort_task = SubprocessTask('sort -i x -o y')` would have been fine as well.
3. In the third line you specify which words in the template string are placeholders for input files. Even if there is just one of them, it must be given as a list.
4. In the fourth line you do similarly to identify the outputs.

Now your task is ready to be run:

In [ ]:
output = sort_task('inputfile.txt')

So what exactly is this thing `output` that is returned by the task? It's not - yet - a file on your filesystem, and quite possibly it never will be.

Instead it's a `FileHandle`, an object that holds the generated content. `FileHandles` have many of the same properties as a Python `Path`, for example they have a `.read_text()` method that returns their content:

In [ ]:
print(output.read_text())

Since the objective of calling this tool from Python was presumably to avoid having to do things directly through files on your filesystem, this may be as much as you need to take forward. But if for some reason you do want to write the results to a genuine file, you can do this:

In [ ]:
output.save('task_output.txt')

You will see a file with this name appear in your file browser pane (to the left).

But you had to start from a real file in your filesystem, right?

No, you don't have to. As well as returning outputs as `FileHandles`, `crossflow` can take `FileHandles` as inputs. Here's an example, you will see how it exploits the "path-like" nature of a `FileHandle` for this:

In [ ]:
from crossflow.filehandling import FileHandler
fh = FileHandler()
inp = fh.create('myinput.txt')
inp.write_text('Here is \nanother file \nwith some \ntext in it')
print(inp.read_text())

1. In line 1 we import the `FileHandler` class.
2. In line 2 we create an instance of a `FileHandler`
3. In line 3 we create a `FileHandle`, called `inp` using the .create() method of the `FileHandler`. This takes a 'placeholder' name for the `FileHandle` - it doesn't have to be the real name of a file and you will see no file with that name is created. It is useful however to give it a reasonable name - in particular, giving it a sensible extension (here ".txt") can be helpful later on.
4. In line 4 we put some data into `inp`, using its .write_text() method.
5. In line 5 we check that we have created what we wanted.

Now let's take it for a run:

In [ ]:
myoutput = sort_task(inp)
print(myoutput.read_text())

## Working with multiple inputs/outputs

The `sort` program (tool) can take multiple files as inputs, it concatenates them and then puts the sorted lines from them all into the output file. To allow our task to do the same, we need to rewrite the template slightly.

Imagine we know there will always be two input files:

In [ ]:
double_sort = SubprocessTask('sort -i input1.txt input2.txt -o sorted.txt')
double_sort.set_inputs(['input1.txt', 'input2.txt'])
double_sort.set_outputs(['sorted.txt'])
double_sorted = double_sort('inputfile.txt', inp) # can mix real files and filehandles, no problem
print(double_sorted.read_text())

But what if you want be able to process an arbitrary number of input files? Crossflow can handle that too:

In [ ]:
multi_sort = SubprocessTask('sort -i input*.txt -o sorted.txt')
multi_sort.set_inputs(['input*.txt'])
multi_sort.set_outputs(['sorted.txt'])
multi_sorted = multi_sort([inp, 'inputfile.txt', inp]) # note input is a single argument, but a list
print(multi_sorted.read_text())

## Beyond files

Quite often your tool needs not just input files, but the setting of other parameters. For example, `sort` can take an argument `--sort=<method>` where <method> is "numeric" or "random", to change the sorting method. We can create crossflow tasks that accomodate that quite easily:

In [ ]:
better_sort = SubprocessTask('sort -i input*.txt --sort={method} -o sorted.txt') # non-filename arguments are indicated by {braces}
better_sort.set_inputs(['input*.txt', 'method'])
better_sort.set_outputs(['sorted.txt'])
inputfiles = [inp, 'inputfile.txt', inp]
better_sorted = better_sort(inputfiles, 'random')
print(better_sorted.read_text())

## Debugging

Of course, very occasionally (!) something will go wrong. To start with, what happens if we try to run the original `better_sort` with an unsupported sorting method:

In [ ]:
result = better_sort('inputfile.txt', 'bogus')

Hmm - that's not very easy to interpret. In such situations, it can be worth slightly re-writing the task, so errors are captured more gracefully:

In [ ]:
debug_sort = SubprocessTask('sort -i input*.txt --sort={method} -o sorted.txt')
debug_sort.set_inputs(['input*.txt', 'method'])
debug_sort.set_outputs(['sorted.txt', 'DEBUGINFO']) # Note new output with special name
inputfiles = [inp, 'inputfile.txt', inp]
debug_sorted, debuginfo = debug_sort(inputfiles, 'bogus') # the task returns two outputs now
if debuginfo.returncode == 0: # Happens if there is no error
    print(debug_sorted.read_text())
else:
    print(debuginfo.stderr.decode('utf-8'))

## Summary

`Crossflow` provides an easy and flexible way to quickly interface your Python code with almost any command-line type tool.

The only (major) exception are tools that need interactivity during their run (e.g. raise a prompt asking for further information, etc.)

`Crossflow` can do much more than has been covered here, in particular it can work with [Dask](https://www.dask.org/) to enable compute-heavy tools to be accessed remotely in a distributed computing environment. Read the paper to learn more.